# Object-Oriented Programming

Open the source code of any major quantum computing framework and you will find the same pattern everywhere. In Qiskit, a `QuantumCircuit` is an object. In PennyLane, a `QNode` is an object. In QuTiP, a `Qobj` represents any quantum object from a ket to a Hamiltonian. These frameworks are built on **object-oriented programming**: the practice of bundling data and the functions that operate on it into a single unit.

This notebook accompanies **Lesson 6** of *Python Programming for Quantum Technology I*. You will build a `Qubit` class from scratch, extend it with inheritance, and assemble it into a `QuantumRegister`.

Topics covered:
- Classes and instances
- `__init__` and instance attributes
- Methods and `self`
- Dunder methods (`__str__`, `__repr__`, `__len__`)
- Inheritance and `super()`
- Composing objects into a register

---
## 1. Classes and Instances

A **class** is a blueprint. An **instance** is a concrete object built from that blueprint. Create an instance by calling the class like a function.

In [ ]:
class Qubit:
    pass   # empty class for now

q = Qubit()
print(type(q))   # <class '__main__.Qubit'>
print(isinstance(q, Qubit))   # True

---
## 2. `__init__` and Instance Attributes

`__init__` is called automatically when a new instance is created. It sets up **instance attributes** via `self.attribute = value`.

`self` is the instance being constructed. Python passes it automatically; you never write it in a call.

In [ ]:
import math

class Qubit:
    def __init__(self, alpha=1.0+0j, beta=0.0+0j):
        """
        Initialize a qubit with complex amplitudes for |0⟩ and |1⟩.
        The state is normalized automatically.
        """
        norm = math.sqrt(abs(alpha)**2 + abs(beta)**2)
        self.alpha = alpha / norm
        self.beta  = beta  / norm

# Create instances
q0 = Qubit()                        # defaults: |0⟩
q1 = Qubit(alpha=0, beta=1)         # |1⟩
qp = Qubit(alpha=1, beta=1)         # |+⟩ (normalized automatically)
qb = Qubit(alpha=3+0j, beta=4+0j)   # normalized to (0.6, 0.8)

print(f"q0: alpha={q0.alpha}, beta={q0.beta}")
print(f"qb: alpha={qb.alpha}, beta={qb.beta}")
print(f"qp: alpha={qp.alpha:.4f}, beta={qp.beta:.4f}")

### Exercise 2.1

Create a class `ClassicalBit` with an `__init__` that takes a single argument `value` (0 or 1, default 0). It should:
1. Raise a `ValueError` if `value` is not 0 or 1.
2. Store the value as `self.value`.

Create instances for both valid values and verify the error is raised for an invalid input.

In [ ]:
# Your code here


---
## 3. Methods

A **method** is a function defined inside a class. It accesses instance data through `self`.

In [ ]:
import math
import random

class Qubit:
    def __init__(self, alpha=1.0+0j, beta=0.0+0j):
        """Initialize and normalize a qubit state."""
        norm = math.sqrt(abs(alpha)**2 + abs(beta)**2)
        self.alpha = alpha / norm
        self.beta  = beta  / norm

    def prob_zero(self):
        """Return the probability of measuring |0⟩."""
        return abs(self.alpha)**2

    def prob_one(self):
        """Return the probability of measuring |1⟩."""
        return abs(self.beta)**2

    def measure(self):
        """Simulate a projective measurement. Collapses the state."""
        outcome = 0 if random.random() < self.prob_zero() else 1
        if outcome == 0:
            self.alpha, self.beta = 1.0+0j, 0.0+0j
        else:
            self.alpha, self.beta = 0.0+0j, 1.0+0j
        return outcome

    def apply_x(self):
        """Apply the Pauli X (NOT) gate: swap alpha and beta."""
        self.alpha, self.beta = self.beta, self.alpha
        return self   # return self to enable method chaining

    def apply_h(self):
        """Apply the Hadamard gate."""
        s = 1 / math.sqrt(2)
        new_alpha = s * (self.alpha + self.beta)
        new_beta  = s * (self.alpha - self.beta)
        self.alpha = new_alpha
        self.beta  = new_beta
        return self

    def apply_z(self):
        """Apply the Pauli Z gate: negate beta."""
        self.beta = -self.beta
        return self

In [ ]:
random.seed(42)

q = Qubit()   # |0⟩

print(f"Initial: P(|0⟩)={q.prob_zero():.4f}, P(|1⟩)={q.prob_one():.4f}")

# Apply Hadamard: puts qubit into equal superposition
q.apply_h()
print(f"After H: P(|0⟩)={q.prob_zero():.4f}, P(|1⟩)={q.prob_one():.4f}")

# Measure: collapses the state
outcome = q.measure()
print(f"Measured: |{outcome}⟩")
print(f"After measurement: P(|0⟩)={q.prob_zero():.4f}, P(|1⟩)={q.prob_one():.4f}")

In [ ]:
# Method chaining: apply_h and apply_x return self
q2 = Qubit()             # |0⟩
q2.apply_h().apply_z()   # H then Z, chained

print(f"alpha = {q2.alpha:.4f}")
print(f"beta  = {q2.beta:.4f}")
print(f"P(|0⟩) = {q2.prob_zero():.4f}")
print(f"P(|1⟩) = {q2.prob_one():.4f}")

### Exercise 3.1

Add a method `apply_phase(phi)` to the `Qubit` class that applies a phase gate:

$$P(\phi) = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\phi} \end{pmatrix}$$

This leaves `alpha` unchanged and multiplies `beta` by $e^{i\phi}$.

Use `import cmath` for `cmath.exp(1j * phi)`. Return `self` to enable chaining.

Test it: start with $|+\rangle$ (apply H to $|0\rangle$), then apply a phase of $\pi$. Verify that the result is $|-\rangle$ by checking the probabilities and amplitudes.

In [ ]:
import cmath

# Copy the Qubit class above and add apply_phase(self, phi) to it
# Your code here


---
## 4. Dunder Methods

Dunder (double-underscore) methods hook your objects into Python's built-in operations.

In [ ]:
import math
import random

class Qubit:
    def __init__(self, alpha=1.0+0j, beta=0.0+0j):
        norm = math.sqrt(abs(alpha)**2 + abs(beta)**2)
        self.alpha = alpha / norm
        self.beta  = beta  / norm

    def prob_zero(self):
        return abs(self.alpha)**2

    def prob_one(self):
        return abs(self.beta)**2

    def measure(self):
        outcome = 0 if random.random() < self.prob_zero() else 1
        if outcome == 0:
            self.alpha, self.beta = 1.0+0j, 0.0+0j
        else:
            self.alpha, self.beta = 0.0+0j, 1.0+0j
        return outcome

    def apply_x(self):
        self.alpha, self.beta = self.beta, self.alpha
        return self

    def apply_h(self):
        s = 1 / math.sqrt(2)
        new_alpha = s * (self.alpha + self.beta)
        new_beta  = s * (self.alpha - self.beta)
        self.alpha, self.beta = new_alpha, new_beta
        return self

    def apply_z(self):
        self.beta = -self.beta
        return self

    def __str__(self):
        """Human-readable string, used by print()."""
        return (f"Qubit: ({self.alpha:.3f})|0⟩ + ({self.beta:.3f})|1⟩  "
                f"[P(0)={self.prob_zero():.3f}, P(1)={self.prob_one():.3f}]")

    def __repr__(self):
        """Unambiguous representation, used in the interpreter and debugging."""
        return f"Qubit(alpha={self.alpha!r}, beta={self.beta!r})"


q = Qubit(alpha=3, beta=4)
print(q)        # calls __str__
print(repr(q))  # calls __repr__

---
## 5. Inheritance

A **subclass** inherits all attributes and methods of its parent. `super()` calls the parent's version of a method, letting you extend behavior without rewriting it.

A realistic use case: a `NoisyQubit` that behaves like a `Qubit` but has a chance of flipping the measurement result (readout error).

In [ ]:
class NoisyQubit(Qubit):
    def __init__(self, alpha=1.0+0j, beta=0.0+0j, readout_error=0.05):
        """
        A qubit with a probability of readout error on measurement.

        Parameters
        ----------
        readout_error : float
            Probability that the measurement result is flipped.
        """
        super().__init__(alpha, beta)   # call Qubit.__init__
        self.readout_error = readout_error

    def measure(self):
        """Measure with a chance of readout error."""
        true_outcome = super().measure()   # call Qubit.measure
        if random.random() < self.readout_error:
            return 1 - true_outcome        # flip the result
        return true_outcome

    def __str__(self):
        base = super().__str__()
        return base + f"  [readout_error={self.readout_error:.3f}]"


# NoisyQubit inherits prob_zero, prob_one, apply_h, apply_x from Qubit
nq = NoisyQubit(alpha=1, beta=0, readout_error=0.15)
print(nq)
print(f"Is a Qubit?      {isinstance(nq, Qubit)}")
print(f"Is a NoisyQubit? {isinstance(nq, NoisyQubit)}")

In [ ]:
import random
random.seed(7)

# Compare ideal vs noisy qubit prepared in |0⟩ over many measurements
n_trials = 100

ideal_errors = 0
noisy_errors = 0

for _ in range(n_trials):
    # Ideal qubit in |0⟩
    ideal = Qubit(alpha=1, beta=0)
    if ideal.measure() == 1:
        ideal_errors += 1

    # Noisy qubit with 15% readout error rate
    noisy = NoisyQubit(alpha=1, beta=0, readout_error=0.15)
    if noisy.measure() == 1:
        noisy_errors += 1

print(f"Ideal qubit errors:  {ideal_errors}/{n_trials}  ({ideal_errors/n_trials:.2f})")
print(f"Noisy qubit errors:  {noisy_errors}/{n_trials}  ({noisy_errors/n_trials:.2f})")

### Exercise 5.1

Create a `DepolarizingQubit` subclass of `Qubit` that models depolarizing noise. Before measurement, with probability `p_depol`, the qubit's state is replaced by the maximally mixed state: `alpha = beta = 1/√2`.

Requirements:
1. `__init__` takes `alpha`, `beta`, and `p_depol=0.1`. Call `super().__init__`.
2. Override `measure`: with probability `p_depol`, reset the state to equal superposition before calling `super().measure()`.
3. Override `__str__` to include the depolarizing probability.

Test: prepare many qubits in $|0\rangle$ with `p_depol=0.5`. The error rate should be close to 0.5 × 0.5 = 0.25.

In [ ]:
import math
import random

# Your code here


---
## 6. Putting It Together: A Quantum Register

A `QuantumRegister` holds multiple `Qubit` instances and coordinates operations across them. This is the pattern used in every quantum framework.

In [ ]:
import math
import random

class QuantumRegister:
    """A collection of qubits initialized to |0⟩."""

    def __init__(self, n_qubits):
        """Initialize n_qubits qubits, all in the |0⟩ state."""
        self.n_qubits = n_qubits
        self.qubits = [Qubit() for _ in range(n_qubits)]

    def apply_h(self, index):
        """Apply Hadamard to the qubit at index."""
        self.qubits[index].apply_h()
        return self

    def apply_x(self, index):
        """Apply Pauli X to the qubit at index."""
        self.qubits[index].apply_x()
        return self

    def apply_z(self, index):
        """Apply Pauli Z to the qubit at index."""
        self.qubits[index].apply_z()
        return self

    def measure_all(self):
        """Measure all qubits. Returns a list of outcomes."""
        return [q.measure() for q in self.qubits]

    def probabilities(self):
        """Return P(|1⟩) for each qubit."""
        return [q.prob_one() for q in self.qubits]

    def __len__(self):
        return self.n_qubits

    def __str__(self):
        probs = self.probabilities()
        lines = [f"  q{i}: P(|1⟩) = {p:.4f}" for i, p in enumerate(probs)]
        return f"QuantumRegister ({self.n_qubits} qubits):\n" + "\n".join(lines)

In [ ]:
random.seed(42)

# Create a 4-qubit register and put all qubits into superposition
reg = QuantumRegister(4)

for i in range(len(reg)):
    reg.apply_h(i)

print(reg)

outcomes = reg.measure_all()
bitstring = "".join(str(o) for o in outcomes)
print(f"\nMeasurement result: |{bitstring}⟩")

In [ ]:
random.seed(0)

# Run 1000 shots and count bitstring frequencies
n_shots = 1000
counts = {}

for _ in range(n_shots):
    reg = QuantumRegister(3)
    for i in range(3):
        reg.apply_h(i)
    outcomes = reg.measure_all()
    bitstring = "".join(str(o) for o in outcomes)
    counts[bitstring] = counts.get(bitstring, 0) + 1

print(f"{'Bitstring':<12} {'Count':>6} {'Frequency':>10}")
print("-" * 30)
for bits in sorted(counts):
    freq = counts[bits] / n_shots
    print(f"|{bits}⟩ {counts[bits]:>8}   {freq:>9.3f}")

### Exercise 6.1

Extend `QuantumRegister` with a `reset(index)` method that resets the qubit at the given index back to $|0\rangle$, and a `reset_all()` method that resets all qubits.

Then write a simulation that:
1. Creates a 3-qubit register.
2. Applies H to all qubits.
3. Measures all qubits and records the bitstring.
4. Resets all qubits.
5. Repeats steps 2-4 for 500 shots.
6. Prints the count for each bitstring.

Each of the 8 bitstrings should appear roughly 500/8 ≈ 62 times.

In [ ]:
import random
random.seed(1)

# Your code here


---
## Summary

| Concept | Syntax | Notes |
|---------|--------|-------|
| Define a class | `class Name:` | Blueprint for creating objects |
| Create an instance | `obj = Name(args)` | Calls `__init__` automatically |
| Constructor | `def __init__(self, ...):` | Sets up instance attributes |
| Instance attribute | `self.name = value` | Belongs to one specific instance |
| Method | `def method(self, ...):` | `self` is passed automatically; never write it in a call |
| Method chaining | `return self` | Lets you write `obj.a().b().c()` |
| Human-readable string | `def __str__(self):` | Called by `print()` |
| Unambiguous string | `def __repr__(self):` | Called in the interpreter |
| Length | `def __len__(self):` | Called by `len()` |
| Subclass | `class Child(Parent):` | Inherits all parent attributes and methods |
| Call parent method | `super().method(...)` | Extend without rewriting |

**What comes next:** The Python foundations are now complete. The next lesson moves into **NumPy**, the numerical engine that makes quantum simulation practical. Everything written so far with lists of complex numbers will be rewritten with arrays, and the difference in expressiveness and speed will be immediate.

---
## Challenge Problem

**Build a `QuantumCircuit` class.**

Design a `QuantumCircuit` class that records a sequence of gate operations and then executes them on a `QuantumRegister`.

Requirements:
1. `__init__(self, n_qubits)`: store `n_qubits` and initialize an empty list `self.instructions`.
2. `h(self, index)`: append `('H', index)` to `self.instructions`. Return `self`.
3. `x(self, index)`: append `('X', index)`. Return `self`.
4. `z(self, index)`: append `('Z', index)`. Return `self`.
5. `run(self, n_shots=1, seed=None)`: create a `QuantumRegister`, replay all instructions in order, measure all qubits, and return the bitstring. Repeat for `n_shots`, returning a list of bitstrings.
6. `__len__`: return the number of instructions.
7. `__str__`: print the circuit as a numbered list of instructions.

Build and run a 3-qubit circuit that applies H to all qubits, then X to qubit 1, for 10 shots. Print the circuit and the results.

In [ ]:
import math
import random

# Your code here
